In [ ]:
import numpy as np
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
# Load CSV
df = pd.read_csv('amazon.csv')

# Keep only relevant columns
df = df[['review_content', 'rating']].copy()

# Remove rows with non-numeric ratings (e.g., '|')
df = df[pd.to_numeric(df['rating'], errors='coerce').notna()]
df['rating'] = df['rating'].astype(float)

# Drop rows with missing review text
df = df.dropna(subset=['review_content'])

print(f'Total samples after cleaning: {len(df)}')
print(df['rating'].describe())

In [ ]:
# Convert ratings to binary sentiment labels
df['label'] = (df['rating'] >= 4.0).astype(int)

texts = df['review_content'].tolist()
labels = np.array(df['label'].tolist())

print(f'Positive samples: {labels.sum()}')
print(f'Negative samples: {len(labels) - labels.sum()}')
print('\nSample text:', texts[0][:80])
print('Sample label:', labels[0])

In [ ]:
MAX_LEN = 50  # Max words per review (longer will be truncated, shorter padded)
MAX_WORDS = 5000  # Vocabulary size — only keep the top 5000 most frequent words

# Create tokenizer
tokenizer = Tokenizer(num_words=MAX_WORDS)

# Learn vocabulary from all reviews
tokenizer.fit_on_texts(texts)

print(f'Vocabulary size (all unique words): {len(tokenizer.word_index)}')
print('Top 10 words:', list(tokenizer.word_index.items())[:10])

# Convert each review into a sequence of integer IDs
sequences = tokenizer.texts_to_sequences(texts)

print('\nExample review sequence (first 10 IDs):', sequences[0][:10])

In [ ]:
# Pad or truncate all sequences to MAX_LEN
X = pad_sequences(sequences, maxlen=MAX_LEN)

print('X shape:', X.shape)  # (num_samples, MAX_LEN)
print('labels shape:', labels.shape)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, labels, test_size=0.2, random_state=42
)

print(f'Training samples: {len(X_train)}')
print(f'Test samples:     {len(X_test)}')

In [ ]:
model = Sequential()

# Embedding: converts word IDs → dense vectors
model.add(Embedding(
    input_dim=MAX_WORDS,   # Vocabulary size
    output_dim=32,         # Each word becomes a 32-dimensional vector
    input_length=MAX_LEN   # Sequence length
))

# RNN layer: reads words one by one, carries hidden state forward
model.add(SimpleRNN(32))

# Output layer: sigmoid for binary classification
model.add(Dense(1, activation='sigmoid'))

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_test, y_test)
)

In [ ]:
loss, accuracy = model.evaluate(X_test, y_test)
print(f'\nTest Accuracy: {accuracy:.2%}')

In [ ]:
new_review = ["Amazing product, works perfectly and great build quality"]

# Tokenize and pad the new review
new_seq = tokenizer.texts_to_sequences(new_review)
new_padded = pad_sequences(new_seq, maxlen=MAX_LEN)

# Predict
prediction = model.predict(new_padded)
prob = prediction[0][0]

print(f'Prediction probability: {prob:.4f}')
if prob > 0.5:
    print('Sentiment: Positive 😊')
else:
    print('Sentiment: Negative ☹️')